<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 04. DBSCAN: Agrupando por Vecindarios Concurridos
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 10
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/10%20-%20Clustering/Para%20Dummies/04_DBSCAN_Clustering_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del módulo 04 de Clustering. El cuaderno principal habló de "puntos núcleo", "densidad" y "vecindarios `eps`" — aquí vamos a entender la misma idea con una analogía de barrio y el mismo dataset de clientes del centro comercial.

Al terminar podrás explicar, con tus propias palabras:
1. En qué se diferencia DBSCAN de K-Means y del clustering jerárquico.
2. Qué son los puntos núcleo, borde y de ruido.
3. Cómo aplicar DBSCAN en scikit-learn y cómo elegir sus dos parámetros (`eps` y `min_samples`).
4. Por qué DBSCAN puede detectar formas raras (como medias lunas) que K-Means no puede.


---
## 1. La analogía del barrio concurrido 🏘️

Imagina que sobrevuelas una ciudad de noche y quieres identificar "barrios" solo mirando qué tan **iluminadas** (concurridas) están las zonas. No te importa dónde está el "centro" de cada barrio (como en K-Means) — te importa si una casa tiene **suficientes vecinas cerca**.

DBSCAN clasifica cada punto (cada "casa") en una de tres categorías, usando dos reglas: un radio de vecindad `eps` ("qué tan cerca hay que estar para contar como vecino") y un mínimo de vecinos `min_samples` ("cuántos vecinos cercanos hacen falta para considerar la zona concurrida"):

* **Punto núcleo:** tiene al menos `min_samples` vecinos dentro del radio `eps` — vive en una zona bien concurrida.
* **Punto borde:** no tiene tantos vecinos como para ser núcleo él mismo, pero está lo bastante cerca de algún punto núcleo como para "colarse" en su barrio.
* **Punto de ruido:** no cumple ninguna de las dos condiciones anteriores — es una casa aislada en medio de la nada, y DBSCAN la deja **fuera de cualquier barrio**, marcada con la etiqueta especial `-1`.

> 📌 **Para recordar:** a diferencia de K-Means y del clustering jerárquico, DBSCAN es el único de los tres algoritmos del módulo que puede decir "este punto no pertenece a ningún grupo" — no todo el mundo tiene que quedar metido a la fuerza en un barrio.


---
## Configuración del entorno de trabajo 🛠️

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans

df_mall = pd.read_csv('../data/mall_customers.csv')
X = df_mall[['Annual_Income_k', 'Spending_Score']].values

print("Dataset cargado:", df_mall.shape)
df_mall.head()

### 🤔 ¿Qué acaba de pasar?

- Volvemos a cargar `mall_customers.csv`, el mismo dataset de 200 clientes del cuaderno 02, con las mismas dos columnas: ingreso anual (`Annual_Income_k`) y puntaje de gasto (`Spending_Score`).
- A diferencia de K-Means, aquí **no** vamos a estandarizar los datos todavía — el radio `eps` que usaremos más adelante está pensado directamente para la escala original de estas dos columnas (miles de dólares y puntaje de 1 a 100). Si estandarizaras, tendrías que recalcular `eps` desde cero.


---
## 2. Aplicando DBSCAN con scikit-learn ⚡

Los dos parámetros clave que ya conoces de la analogía del barrio:

* `eps`: el radio de vecindad (en las mismas unidades que tus datos).
* `min_samples`: cuántos vecinos (incluyéndose a sí mismo) hacen falta dentro de ese radio para considerar la zona "concurrida".

Probamos con `eps=24` y `min_samples=20` — valores que, como veremos en la sección 3, no salieron de la nada.


In [ ]:
modelo_dbscan = DBSCAN(eps=24, min_samples=20)
grupo_dbscan = modelo_dbscan.fit_predict(X)

print("Clientes marcados como ruido (-1):", (grupo_dbscan == -1).sum())
print("Número de grupos encontrados:", len(set(grupo_dbscan)) - (1 if -1 in grupo_dbscan else 0))
print("\nClientes por grupo:")
print(pd.Series(grupo_dbscan).value_counts().sort_index())

### 🤔 ¿Qué acaba de pasar?

- `fit_predict` le asigna a cada cliente una etiqueta de grupo, igual que hacía K-Means — con la diferencia clave de que aquí la etiqueta `-1` significa "ruido", es decir, "este cliente no encajó en ningún barrio concurrido".
- Con estos parámetros en particular, es posible que DBSCAN encuentre un único gran grupo y ningún (o casi ningún) punto de ruido — recuerda que, a diferencia de K-Means, aquí **no** le dijimos de antemano cuántos grupos queremos: el número de grupos es una *consecuencia* de `eps` y `min_samples`, no un parámetro que se elige directamente.


In [ ]:
plt.figure(figsize=(6.5, 5))
es_ruido = grupo_dbscan == -1

plt.scatter(X[~es_ruido, 0], X[~es_ruido, 1], c=grupo_dbscan[~es_ruido], cmap='viridis', s=30, alpha=0.85)
plt.scatter(X[es_ruido, 0], X[es_ruido, 1], c='red', marker='x', s=60, label='Ruido (-1)')
plt.title("DBSCAN (eps=24, min_samples=20): Clientes del Centro Comercial")
plt.xlabel("Ingreso anual (miles de USD)")
plt.ylabel("Puntaje de gasto (1-100)")
plt.legend()
plt.show()

### 🤔 ¿Qué acaba de pasar?

- Los puntos rojos con "x" son los clientes marcados como ruido — casas aisladas, sin suficientes vecinos cercanos para pertenecer a ningún barrio.
- Compáralo mentalmente con el resultado de K-Means en el cuaderno 02: allí **todos** los 200 clientes quedaban repartidos entre 5 grupos, sin excepción. Aquí, en cambio, DBSCAN se permite decir "esto no encaja en ningún patrón claro" — una capacidad muy útil cuando sospechas que tus datos tienen valores atípicos genuinos.


---
## 3. ¿Cómo elegir `eps` y `min_samples`? 🎯

A diferencia del K de K-Means (que uno simplemente decide de antemano), `eps` y `min_samples` no son tan intuitivos de adivinar a ojo. Una forma sencilla y muy usada de elegir `eps` es el **gráfico de distancia al k-ésimo vecino**:

1. Fijamos `min_samples` (por ejemplo, 20).
2. Para cada cliente, calculamos qué tan lejos está su vecino número 20 más cercano.
3. Ordenamos esas distancias de menor a mayor y las graficamos.

Donde aparece un **codo** claro en esa curva (un salto brusco de "distancias chiquitas" a "distancias grandes"), ese es un buen candidato para `eps`: antes del codo están los clientes bien acompañados (candidatos a núcleo), después del codo empiezan a aparecer los más aislados.


In [ ]:
from sklearn.neighbors import NearestNeighbors

vecinos = NearestNeighbors(n_neighbors=20).fit(X)
distancias, _ = vecinos.kneighbors(X)
distancia_al_vecino_20 = np.sort(distancias[:, -1])

plt.figure(figsize=(6.5, 4.5))
plt.plot(distancia_al_vecino_20)
plt.xlabel("Clientes, ordenados de menor a mayor distancia")
plt.ylabel("Distancia al vecino número 20 más cercano")
plt.title("Gráfico de codo para elegir 'eps' (min_samples=20)")
plt.show()

### 🤔 ¿Qué acaba de pasar?

- La curva empieza plana (muchos clientes con vecinos muy cercanos) y en algún punto se dispara hacia arriba (clientes cada vez más aislados).
- El valor del eje vertical justo donde ocurre ese "codo" es un buen punto de partida para `eps` — de ahí salió el valor `eps=24` que usamos en la sección 2. Como con el método del codo de K-Means, esto es una guía visual, no una fórmula exacta: vale la pena probar un par de valores cercanos al codo y comparar los resultados.


---
## 4. La gran fortaleza de DBSCAN: formas raras 🌙

Volvamos al ejemplo de las dos medias lunas entrelazadas que vimos en el cuaderno 02, donde K-Means fallaba. Comparemos, lado a lado, cómo se comportan K-Means y DBSCAN sobre exactamente los mismos datos.


In [ ]:
from sklearn.datasets import make_moons

X_lunas, _ = make_moons(n_samples=300, noise=0.06, random_state=42)

kmeans_lunas = KMeans(n_clusters=2, n_init=10, random_state=42)
grupo_kmeans_lunas = kmeans_lunas.fit_predict(X_lunas)

dbscan_lunas = DBSCAN(eps=0.2, min_samples=5)
grupo_dbscan_lunas = dbscan_lunas.fit_predict(X_lunas)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].scatter(X_lunas[:, 0], X_lunas[:, 1], c=grupo_kmeans_lunas, cmap='viridis', s=25, alpha=0.85)
axes[0].set_title("K-Means: corta las lunas por la mitad", fontweight='bold')

es_ruido_lunas = grupo_dbscan_lunas == -1
axes[1].scatter(X_lunas[~es_ruido_lunas, 0], X_lunas[~es_ruido_lunas, 1],
                 c=grupo_dbscan_lunas[~es_ruido_lunas], cmap='viridis', s=25, alpha=0.85)
axes[1].scatter(X_lunas[es_ruido_lunas, 0], X_lunas[es_ruido_lunas, 1], c='red', marker='x', label='Ruido')
axes[1].set_title("DBSCAN: recupera las dos lunas correctamente", fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

### 🤔 ¿Qué acaba de pasar?

- K-Means solo entiende "distancia al centro más cercano", así que corta las dos medias lunas con una línea más o menos recta, sin importarle su forma real.
- DBSCAN, en cambio, va "caminando" por las zonas densamente conectadas — sigue el contorno curvo de cada media luna en vez de mirar un único centro — y logra separarlas correctamente, siempre que `eps` esté bien calibrado para la escala de estos datos.
- Esta es la razón principal para usar DBSCAN: cuando sospechas que tus grupos **no** tienen forma redondeada, o cuando esperas que existan valores atípicos genuinos que no deberían forzarse dentro de ningún grupo.


---
## 5. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| DBSCAN | Agrupa según qué tan "concurrida" (densa) está la vecindad de cada punto, no según distancia a un centro. |
| `eps` | El radio de vecindad: qué tan cerca hay que estar para contar como vecino. |
| `min_samples` | Cuántos vecinos cercanos hacen falta para que una zona se considere densa. |
| Punto núcleo / borde / ruido | Vive en zona densa / está pegado a una zona densa / está aislado (etiqueta `-1`). |
| Ventaja clave | No necesitas fijar el número de grupos de antemano, y puede detectar formas irregulares y valores atípicos genuinos. |
| Gráfico de codo (k-distance) | Herramienta visual para elegir un buen valor de `eps`. |

➡️ **Siguiente paso:** el cuaderno [05 - Evaluación, Selección de K y Benchmark (Para Dummies)](05_Evaluacion_Seleccion_K_y_Benchmark_Dummies.ipynb) cierra el módulo respondiendo una pregunta clave que quedó pendiente en los tres cuadernos anteriores: si no tenemos etiquetas verdaderas, ¿cómo sabemos si un agrupamiento es "bueno"?


---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>
